In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import vectorbt as vbt

In [ ]:
# Download the data
data = yf.download("SPY", start="2022-01-01", end="2025-10-26", auto_adjust=True)
data.columns = data.columns.droplevel(1)

# Calculate the SMA
data["SMA50"] = data["Close"].rolling(window=50).mean()

# Determine the position state (1 for long, 0 for flat)
data["Position"] = np.where(data["Close"] > data["SMA50"], 1, 0)

# Calculate the signal (1 for buy, -1 for sell, 0 for hold)
data["Signal"] = data["Position"].diff()

# Display the rows where a trade signal occurred
print("Trade Signals (1 = Buy, -1 = Sell):")
print(data[data["Signal"] != 0].head())

[*********************100%***********************]  1 of 1 completed

Trade Signals (1 = Buy, -1 = Sell):
Price            Close        High         Low        Open     Volume  \
Date                                                                    
2022-01-03  453.210358  453.343192  449.548334  451.872667   72668200   
2022-03-18  423.032654  423.356215  416.085535  416.827830  106345500   
2022-04-11  418.655029  423.489458  418.150650  422.642465   89770500   
2022-04-13  421.881165  422.642482  416.675574  416.856392   74070400   
2022-04-14  416.627991  423.232525  416.523293  422.109542   97869500   

Price            SMA50  Position  Signal  
Date                                      
2022-01-03         NaN         0     NaN  
2022-03-18  419.555723         1     1.0  
2022-04-11  419.390375         0    -1.0  
2022-04-13  419.038950         1     1.0  
2022-04-14  418.693618         0    -1.0  


In [3]:
# Create entry signals where Signal is 1 (buy)
entries = data["Signal"] == 1

# Create exit signals where Signal is -1 (sell)
exits = data["Signal"] == -1

In [4]:
# Run the Vectorbt backtest
portfolio = vbt.Portfolio.from_signals(
    close=data["Close"],
    entries=entries,
    exits=exits,
    init_cash=100_000,  # Start with $100,000
    freq="D",  # Use daily frequency for calculations
)

# Print the performance statistics
print("\n--- Backtest Performance ---")
print(portfolio.stats())


--- Backtest Performance ---
Start                                2022-01-03 00:00:00
End                                  2025-10-27 00:00:00
Period                                 958 days 00:00:00
Start Value                                     100000.0
End Value                                  138661.256895
Total Return [%]                               38.661257
Benchmark Return [%]                           50.858206
Max Gross Exposure [%]                             100.0
Total Fees Paid                                      0.0
Max Drawdown [%]                               20.259819
Max Drawdown Duration                  380 days 00:00:00
Total Trades                                          28
Total Closed Trades                                   27
Total Open Trades                                      1
Open Trade PnL                              26044.813027
Win Rate [%]                                   25.925926
Best Trade [%]                                  16.88075
W

In [5]:
# Plot the closing price first to create the base figure
fig = data["Close"].vbt.plot(trace_kwargs=dict(name="Price"))

# Add the SMA50 indicator to the same figure
data["SMA50"].vbt.plot(fig=fig, trace_kwargs=dict(name="SMA50"))

# Add the buy/sell signals from the portfolio to the figure
portfolio.positions.plot(fig=fig)

fig.show()

## Analyze Our Results

* **Total Return vs. Benchmark:** Our strategy returned 37.35%, while simply buying and holding SPY over the same period would have returned 49.43%.

    * **Our strategy, in its current form, did not beat the market.**

* **Win Rate (25.9%):** This is very low. It means we lost on roughly 3 out of every 4 trades. This isn't necessarily a dealbreaker for a trend-following strategy, if the wins are massive and the losses are small. Our "Profit Factor" of 1.48 (total money won / total money lost) shows that the wins did outweigh the losses, but not by a huge margin.

* **Max Drawdown (20.2%):** A 20% drop in our portfolio value is significant. This happened during a period in the market (Exit Dec 2022) that was choppy and sideways where the price frequently crisscrossed the 50-day moving average, generating a losing "whipsaw" trade.

* **Sharpe Ratio (0.99):** This is a measure of risk-adjusted return. A Sharpe ratio near 1.0 is generally considered pretty good. It suggests that while the total return was lower, we got a decent return for the amount of volatility (risk) we took on.

### Hypothesis

**The Signal is Too Simplistic**

Using just the price crossing a single moving average generates many false signals in sideways markets. Instead we can evolve to using two moving average signals, a short window and a long window. This was already implemented by Henry.

In [6]:
# Calculate the SMA
data["SMA100"] = data["Close"].rolling(window=100).mean()

# Determine the position state (1 for long, 0 for flat)
data["Position"] = np.where(data["SMA50"] > data["SMA100"], 1, 0)

# Calculate the signal (1 for buy, -1 for sell, 0 for hold)
data["Signal"] = data["Position"].diff()

# Display the rows where a trade signal occurred
print("Trade Signals (1 = Buy, -1 = Sell):")
print(data[data["Signal"] != 0].head())

Trade Signals (1 = Buy, -1 = Sell):
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2022-01-03  453.210358  453.343192  449.548334  451.872667  72668200   
2022-09-09  388.617157  389.486914  384.660244  384.927859  76706900   
2022-10-17  352.036682  353.149936  342.881148  349.339931  93168200   
2022-12-30  368.702972  368.847582  364.846549  366.977244  84022200   
2023-10-17  425.035156  427.101771  421.555119  421.906035  75324700   

Price            SMA50  Position  Signal      SMA100  
Date                                                  
2022-01-03         NaN         0     NaN         NaN  
2022-09-09  384.386738         1     1.0  383.801830  
2022-10-17  374.883465         0    -1.0  375.119959  
2022-12-30  373.699638         1     1.0  373.510956  
2023-10-17  427.054237         0    -1.0  427.214890  


In [7]:
entries = data["Signal"] == 1
exits = data["Signal"] == -1

portfolio_dual_sma = vbt.Portfolio.from_signals(
    close=data["Close"], entries=entries, exits=exits, init_cash=100_000, freq="D"
)

# Print the performance statistics
print("\n--- Dual SMA (50/100) Backtest Performance ---")
print(portfolio_dual_sma.stats())


--- Dual SMA (50/100) Backtest Performance ---
Start                         2022-01-03 00:00:00
End                           2025-10-27 00:00:00
Period                          958 days 00:00:00
Start Value                              100000.0
End Value                           144408.843512
Total Return [%]                        44.408844
Benchmark Return [%]                    50.858206
Max Gross Exposure [%]                      100.0
Total Fees Paid                               0.0
Max Drawdown [%]                        12.883022
Max Drawdown Duration           181 days 00:00:00
Total Trades                                    4
Total Closed Trades                             3
Total Open Trades                               1
Open Trade PnL                       16602.058334
Win Rate [%]                            66.666667
Best Trade [%]                          22.388256
Worst Trade [%]                         -9.412985
Avg Winning Trade [%]                   18.833365
Av

**The dual SMA crossover is a significant improvement over the single SMA strategy.**

The results confirm our hypothesis:

* Total Return increased from 37.35% to 43.05%.

* Sharpe Ratio improved from 0.99 to 1.08, indicating better risk-adjusted returns.

* Most impressively, the Max Drawdown was nearly cut in half, from 20.26% to 12.88%. This is a huge improvement in risk management.

* The Win Rate jumped from 26% to 67%.

In [8]:
windows = np.arange(2, 201, 2)
fast_ma, slow_ma = vbt.MA.run_combs(
    data["Close"], window=windows, r=2, short_names=["fast", "slow"]
)
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

combination_testing_portfolio = vbt.Portfolio.from_signals(
    close=data["Close"], entries=entries, exits=exits, init_cash=100_000, freq="D"
)

sharpe_ratios = combination_testing_portfolio.sharpe_ratio()
best_params = sharpe_ratios.idxmax()
best_fast_window, best_slow_window = best_params

print(f"\nBest Parameters Found:")
print(f"  Fast Window: {best_fast_window}")
print(f"  Slow Window: {best_slow_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    xaxis_title="Slow Window",
    yaxis_title="Fast Window",
    title="In-Sample Sharpe Ratios for Dual SMA Strategy",
).show()


Best Parameters Found:
  Fast Window: 20
  Slow Window: 200
  Resulting Sharpe Ratio: 1.74


d:\Development\TigerQuant\demos\.venv\Lib\site-packages\jupyter_client\session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



In [9]:
def dual_sma_strategy(close_price, fast_window=50, slow_window=100):
    fast_sma = close_price.rolling(window=fast_window).mean()
    slow_sma = close_price.rolling(window=slow_window).mean()
    entries = (fast_sma > slow_sma) & (fast_sma.shift(1) <= slow_sma.shift(1))
    exits = (fast_sma < slow_sma) & (fast_sma.shift(1) >= slow_sma.shift(1))
    return entries, exits


def single_sma_strategy(close_price, window=50):
    sma = close_price.rolling(window=window).mean()
    entries = (close_price > sma) & (close_price.shift(1) <= sma.shift(1))
    exits = (close_price < sma) & (close_price.shift(1) >= sma.shift(1))
    return entries, exits

In [10]:
entries, exits = dual_sma_strategy(data["Close"], 50, 100)

test = vbt.Portfolio.from_signals(
    close=data["Close"], entries=entries, exits=exits, init_cash=100_000, freq="D"
)
test.stats()

Start                         2022-01-03 00:00:00
End                           2025-10-27 00:00:00
Period                          958 days 00:00:00
Start Value                              100000.0
End Value                           144408.843512
Total Return [%]                        44.408844
Benchmark Return [%]                    50.858206
Max Gross Exposure [%]                      100.0
Total Fees Paid                               0.0
Max Drawdown [%]                        12.883022
Max Drawdown Duration           181 days 00:00:00
Total Trades                                    4
Total Closed Trades                             3
Total Open Trades                               1
Open Trade PnL                       16602.058334
Win Rate [%]                            66.666667
Best Trade [%]                          22.388256
Worst Trade [%]                         -9.412985
Avg Winning Trade [%]                   18.833365
Avg Losing Trade [%]                    -9.412985


"The goal of this analysis is to prevent **overfitting**. We tune the strategy on one historical period (in-sample) and then validate it on a completely separate, unseen period (out-of-sample) to see if the performance is real or just a fluke.

In [11]:
in_sample = data[data.index < "2025-01-01"]
out_sample = data[data.index >= "2025-01-01"]

single_sma_entries, single_sma_exits = single_sma_strategy(data["Close"], window=50)
dual_sma_50_100_entries, dual_sma_50_100_exits = dual_sma_strategy(
    data["Close"], fast_window=50, slow_window=100
)
dual_sma_20_200_entries, dual_sma_20_200_exits = dual_sma_strategy(
    data["Close"], fast_window=20, slow_window=200
)

all_entries = pd.concat(
    {
        "Single_SMA_50": single_sma_entries,
        "SMA_50_100": dual_sma_50_100_entries,
        "SMA_20_200": dual_sma_20_200_entries,
    },
    axis=1,
)  # axis=1 means to stack horizontally (add more columns)

all_exits = pd.concat(
    {
        "Single_SMA_50": single_sma_exits,
        "SMA_50_100": dual_sma_50_100_exits,
        "SMA_20_200": dual_sma_20_200_exits,
    },
    axis=1,
)

in_entries = all_entries.loc[in_sample.index]
in_exits = all_exits.loc[in_sample.index]
out_entries = all_entries.loc[out_sample.index]
out_exits = all_exits.loc[out_sample.index]


print("--IN-SAMPLE PERFORMANCE--")
in_sample_portfolio = vbt.Portfolio.from_signals(
    close=in_sample["Close"],
    entries=in_entries,
    exits=in_exits,
    init_cash=100_000,
    freq="D",
)

display(
    in_sample_portfolio.stats(agg_func=None)
)  # Use agg_func=None to stop vectorbt from taking the average of all strategies

--IN-SAMPLE PERFORMANCE--


,Start,End,Period,Start Value,End Value,Total Return [%],Benchmark Return [%],Max Gross Exposure [%],Total Fees Paid,Max Drawdown [%],...,Avg Winning Trade [%],Avg Losing Trade [%],Avg Winning Trade Duration,Avg Losing Trade Duration,Profit Factor,Expectancy,Sharpe Ratio,Calmar Ratio,Omega Ratio,Sortino Ratio
Single_SMA_50,2022-01-03,2024-12-31,753 days,100000.0,113321.791554,13.321792,28.194143,100.0,0.0,20.259819,...,5.873113,-1.452775,55 days 06:51:25.714285714,5 days 02:39:59.999999999,1.528005,532.871662,0.523038,0.308472,1.097635,0.720775
SMA_50_100,2022-01-03,2024-12-31,753 days,100000.0,132077.289658,32.077290,28.194143,100.0,0.0,12.883022,...,15.278473,-9.412985,199 days 00:00:00,26 days 00:00:00,1.470343,2213.663760,1.049997,1.120669,1.202536,1.510354
SMA_20_200,2022-01-03,2024-12-31,753 days,100000.0,148545.587105,48.545587,28.194143,100.0,0.0,9.974305,...,NaN,NaN,NaT,NaT,NaN,NaN,1.626628,2.119939,1.322949,2.410217


As expected, on the data we used for tuning, our tuned `SMA_20_200` strategy looks like a superstar. It has the highest Sharpe Ratio. This is its home turf. 

Now for the real test. We take our winning parameters and run them on the 2025 data. This is the moment of truth.

In [12]:
print("\n--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---")
out_sample_portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"],
    entries=out_entries,
    exits=out_exits,
    init_cash=100_000,
    freq="D",
)

display(out_sample_portfolio.stats(agg_func=None))


--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---


,Start,End,Period,Start Value,End Value,Total Return [%],Benchmark Return [%],Max Gross Exposure [%],Total Fees Paid,Max Drawdown [%],...,Avg Winning Trade [%],Avg Losing Trade [%],Avg Winning Trade Duration,Avg Losing Trade Duration,Profit Factor,Expectancy,Sharpe Ratio,Calmar Ratio,Omega Ratio,Sortino Ratio
Single_SMA_50,2025-01-02,2025-10-27,205 days,100000.0,122360.628961,22.360629,17.969322,100.0,0.0,2.984648,...,NaN,-0.311388,NaT,13 days,0.0,-311.214497,3.072992,14.485333,1.663976,4.975667
SMA_50_100,2025-01-02,2025-10-27,205 days,100000.0,112989.966308,12.989966,17.969322,100.0,0.0,2.984648,...,NaN,NaN,NaT,NaT,NaN,NaN,2.780193,8.138377,1.840601,4.302489
SMA_20_200,2025-01-02,2025-10-27,205 days,100000.0,118738.647465,18.738647,17.969322,100.0,0.0,2.984648,...,NaN,NaN,NaT,NaT,NaN,NaN,3.384012,11.985343,1.943979,5.560324


How much did the performance **degrade**?

`SMA_20_200`'s return dropped significantly from 48% toi 17%, however, the length of in_sample was WAY longer than the out of sample time period. It still outperformed the benchmark return of buying and holding SPY for the entire period. However, it dropped behind our OG single SMA strategy. 

One thing that should be noted is the sharpe ratio of our tuned `SMA_20_200' strategy. All of our sharpe's are quite high, but 3.2 is ridiculous, especially considered the strategy also outpreformed our benchmark. This means our trading strategy was getting the same/better return for much less risk.

The max drawdown in each of our startegies was the same. Interesting. However during our in sample testing, our tuned strategy showed that it limited drawdown way better than the pther strategies. 

In [13]:
out_sample_portfolio["SMA_20_200"].plot().show()

Some questions we should look for when analyzing a strategy more deeply are: 

* **Drawdowns**: When did the biggest drop in portfolio value happen? Was the market trending down, or was it choppy and moving sideways?

* **Missed Opportunities**: Were there strong uptrends that your strategy was too slow to enter? Why?

* **Losing Trades:** Look at the losing trades. What did the market look like? Was it a "whipsaw" (price quickly reversed after you entered)?

In [14]:
records = out_sample_portfolio.get_trades().records_readable

display(records[records["Column"] == "SMA_20_200"])

,Exit Trade Id,Column,Size,Entry Timestamp,Avg Entry Price,Entry Fees,Exit Timestamp,Avg Exit Price,Exit Fees,PnL,Return,Direction,Status,Position Id
4,4,SMA_20_200,173.669411,2025-05-23,575.806641,0.0,2025-10-27,683.705017,0.0,18738.647465,0.187386,Long,Open,4


Literally just one trade lol.

What would've been the best window combination on the out of sample data and how did it perform comapared to itself in sample?

In [15]:
windows = np.arange(2, 201, 2)
fast_ma, slow_ma = vbt.MA.run_combs(
    out_sample["Close"], window=windows, r=2, short_names=["fast", "slow"]
)
entries = fast_ma.ma_crossed_above(slow_ma)
exits = fast_ma.ma_crossed_below(slow_ma)

combination_testing_portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"], entries=entries, exits=exits, init_cash=100_000, freq="D"
)

sharpe_ratios = combination_testing_portfolio.sharpe_ratio()
best_params = sharpe_ratios.idxmax()
best_fast_window, best_slow_window = best_params

print(f"\nBest Parameters Found:")
print(f"  Fast Window: {best_fast_window}")
print(f"  Slow Window: {best_slow_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    xaxis_title="Slow Window",
    yaxis_title="Fast Window",
    title="Out of Sample Sharpe Ratios for Dual SMA Strategy",
).show()


Best Parameters Found:
  Fast Window: 2
  Slow Window: 90
  Resulting Sharpe Ratio: inf


d:\Development\TigerQuant\demos\.venv\Lib\site-packages\jupyter_client\session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant: nan
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



And for just single SMA on out of sample?

In [16]:
windows = np.arange(2, 201, 2)
sma = vbt.MA.run(out_sample["Close"], window=windows, short_name="sma")
price = out_sample["Close"].vbt.tile(len(windows))
price.columns = sma.ma.columns
entries = price > sma.ma
exits = price < sma.ma

portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"], entries=entries, exits=exits, init_cash=100_000, freq="D"
)

sharpe_ratios = portfolio.sharpe_ratio()
best_param = sharpe_ratios.idxmax()
best_window = best_param

print(f"\nBest Parameter Found:")
print(f"  Window: {best_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    title="Out of Sample Sharpe Ratios for Single SMA Strategy"
).show()


Best Parameter Found:
  Window: 44
  Resulting Sharpe Ratio: 3.59


Single SMA in sample?

In [17]:
windows = np.arange(2, 201, 2)
sma = vbt.MA.run(in_sample["Close"], window=windows, short_name="sma")
price = in_sample["Close"].vbt.tile(len(windows))
price.columns = sma.ma.columns
entries = price > sma.ma
exits = price < sma.ma

portfolio = vbt.Portfolio.from_signals(
    close=in_sample["Close"], entries=entries, exits=exits, init_cash=100_000, freq="D"
)

sharpe_ratios = portfolio.sharpe_ratio()
best_param = sharpe_ratios.idxmax()
best_window = best_param

print(f"\nBest Parameter Found:")
print(f"  Window: {best_window}")
print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

sharpe_ratios.vbt.heatmap(
    title="In Sample Sharpe Ratios for Single SMA Strategy"
).show()


Best Parameter Found:
  Window: 200
  Resulting Sharpe Ratio: 1.40


Let's test all strategies against each other now and print the stats.

In [18]:
in_sample = data[data.index < "2025-01-01"]
out_sample = data[data.index >= "2025-01-01"]

sma_50_entries, sma_50_exits = single_sma_strategy(data["Close"], window=50)
sma_44_entries, sma_44_exits = single_sma_strategy(data["Close"], window=44)
sma_200_entries, sma_200_exits = single_sma_strategy(data["Close"], window=200)

sma_50_100_entries, sma_50_100_exits = dual_sma_strategy(
    data["Close"], fast_window=50, slow_window=100
)
sma_20_200_entries, sma_20_200_exits = dual_sma_strategy(
    data["Close"], fast_window=20, slow_window=200
)
sma_2_90_entries, sma_2_90_exits = dual_sma_strategy(
    data["Close"], fast_window=2, slow_window=90
)

all_entries = pd.concat(
    {
        "Single_SMA_50": sma_50_entries,
        "Single_SMA_44": sma_44_entries,
        "Single_SMA_200": sma_200_entries,
        "SMA_50_100": sma_50_100_entries,
        "SMA_20_200": sma_20_200_entries,
        "SMA_2_90": sma_2_90_entries,
    },
    axis=1,
)  # axis=1 means to stack horizontally (add more columns)

all_exits = pd.concat(
    {
        "Single_SMA_50": sma_50_exits,
        "Single_SMA_44": sma_44_exits,
        "Single_SMA_200": sma_200_exits,
        "SMA_50_100": sma_50_100_exits,
        "SMA_20_200": sma_20_200_exits,
        "SMA_2_90": sma_2_90_exits,
    },
    axis=1,
)

in_entries = all_entries.loc[in_sample.index]
in_exits = all_exits.loc[in_sample.index]
out_entries = all_entries.loc[out_sample.index]
out_exits = all_exits.loc[out_sample.index]


print("--IN-SAMPLE PERFORMANCE--")
in_sample_portfolio = vbt.Portfolio.from_signals(
    close=in_sample["Close"],
    entries=in_entries,
    exits=in_exits,
    init_cash=100_000,
    freq="D",
)

display(
    in_sample_portfolio.stats(agg_func=None)[
        ["Total Return [%]", "Benchmark Return [%]", "Max Drawdown [%]", "Sharpe Ratio"]
    ]
)  # Use agg_func=None to stop vectorbt from taking the average of all strategies

print("\n--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---")
out_sample_portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"],
    entries=out_entries,
    exits=out_exits,
    init_cash=100_000,
    freq="D",
)

display(
    out_sample_portfolio.stats(agg_func=None)[
        ["Total Return [%]", "Benchmark Return [%]", "Max Drawdown [%]", "Sharpe Ratio"]
    ]
)

--IN-SAMPLE PERFORMANCE--


,Total Return [%],Benchmark Return [%],Max Drawdown [%],Sharpe Ratio
Single_SMA_50,13.321792,28.194143,20.259819,0.523038
Single_SMA_44,19.598768,28.194143,17.718059,0.727253
Single_SMA_200,40.347444,28.194143,9.061750,1.403405
SMA_50_100,32.077290,28.194143,12.883022,1.049997
SMA_20_200,48.545587,28.194143,9.974305,1.626628
SMA_2_90,29.931028,28.194143,13.745002,1.019004



--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---


,Total Return [%],Benchmark Return [%],Max Drawdown [%],Sharpe Ratio
Single_SMA_50,22.360629,17.969322,2.984648,3.072992
Single_SMA_44,22.570596,17.969322,2.984648,3.096042
Single_SMA_200,16.820438,17.969322,2.984648,2.886789
SMA_50_100,12.989966,17.969322,2.984648,2.780193
SMA_20_200,18.738647,17.969322,2.984648,3.384012
SMA_2_90,18.527797,17.969322,5.183911,2.745567


**The best-performing strategy in-sample is rarely the best-performing strategy out-of-sample.** This is the perfect demonstration of overfitting.

Let's discuss...

1. **In-Sample: The "Perfect" Strategy Emerges.** During the training period (before 2025), the `SMA_20_200` strategy was the undisputed champion. It crushed the benchmark return (48% vs 28%), had the lowest drawdown (under 10%), and the highest Sharpe Ratio (1.63). If we stopped here, we would confidently declare it the best strategy. The other dual SMA strategies also beat the benchmark, confirming that our hypothesis  was a good one. The single SMA strategies were clear underperformers.

2. **Out-of-Sample: The Great Reversal.** This is the crucial part. When tested on unseen 2025 data, the story completely flipped:

    * Our in-sample champion, `SMA_20_200`, only barely beat the benchmark (17.6% vs 16.9%). While its Sharpe Ratio was high, its actual profit advantage was minimal.

    * The simple `Single_SMA_44` and `Single_SMA_50` strategies, which were the worst performers in-sample, suddenly became the best performers out-of-sample, beating the benchmark by a healthy margin (~21% vs 16.9%).

    * **The Lesson:** We have perfectly demonstrated **overfitting**. The `SMA_20_200` was so perfectly tuned to the specific market conditions of 2022-2024 that its predictive power failed on new data. The simpler, less "optimized" models proved to be more robust and adaptable.

In [19]:
display(out_sample_portfolio.stats(agg_func=None).T)

,Single_SMA_50,Single_SMA_44,Single_SMA_200,SMA_50_100,SMA_20_200,SMA_2_90
Start,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00
End,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00
Period,205 days 00:00:00,205 days 00:00:00,205 days 00:00:00,205 days 00:00:00,205 days 00:00:00,205 days 00:00:00
Start Value,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
End Value,122360.628961,122570.596217,116820.437645,112989.966308,118738.647465,118527.796625
Total Return [%],22.360629,22.570596,16.820438,12.989966,18.738647,18.527797
Benchmark Return [%],17.969322,17.969322,17.969322,17.969322,17.969322,17.969322
Max Gross Exposure [%],100.0,100.0,100.0,100.0,100.0,100.0
Total Fees Paid,0.0,0.0,0.0,0.0,0.0,0.0
Max Drawdown [%],2.984648,2.984648,2.984648,2.984648,2.984648,5.183911


For every strategy except `SMA_2_90`, the Avg winning trade [%] and avg winning trade duration are NaN and NaT respectively. Why is this? These strategies don't have enough time to exit their positions in the short out of sample time frame, so the insane profit we see is just calculated from the remaining open position. We've reached a point where the limitations of our test are influencing the results. 

The best next step is to **pull more historical data**. Our current 10-month out-of-sample period is too short to be statistically significant, as we've noticed with the lack of closed trades. Another hypothesis to test would be to cap the window sizes in order to promote more crossovers, therefore more signals and more trades. However, before we can determine if "faster windows" are better, we need a more robust baseline to test against.

In [ ]:
data = yf.download("SPY", start="2007-01-01", end="2025-10-26", auto_adjust=True)
data

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2007-01-03,99.688599,100.739292,99.124478,100.309144,94807600
2007-01-04,99.900146,100.168111,99.152678,99.589874,69620600
2007-01-05,99.103294,99.709733,98.990477,99.660377,76645300
2007-01-08,99.561707,99.716843,98.898853,99.300800,71655000
2007-01-09,99.477043,99.850778,99.004576,99.646275,75680100
...,...,...,...,...,...
2025-10-21,671.289978,672.989990,669.979980,671.440002,56249000
2025-10-22,667.799988,672.000000,663.299988,672.000000,80564000


With a much larger dataset, we can create a more meaningful split. A good standard is roughly an 80/20 split.

This gives us a long 15-year training period and a nearly 4-year validation period that includes the volatility of recent years.

In [21]:
in_sample_end = "2021-12-31"

in_sample = data[data.index <= in_sample_end]
out_sample = data[data.index > in_sample_end]

Now, let's find the best performing hyperparameters for both single and dual SMA strategies on both the in sample and out of sample datasets.

In [22]:
def find_best_dual_sma_params(close, min_window_size=2, max_window_size=200, step=2):
    windows = np.arange(min_window_size, max_window_size + 1, step)
    fast_ma, slow_ma = vbt.MA.run_combs(
        close, window=windows, r=2, short_names=["fast", "slow"]
    )
    entries = fast_ma.ma_crossed_above(slow_ma)
    exits = fast_ma.ma_crossed_below(slow_ma)

    portfolio = vbt.Portfolio.from_signals(
        close=close, entries=entries, exits=exits, init_cash=100_000, freq="D"
    )

    sharpe_ratios = portfolio.sharpe_ratio()
    best_params = sharpe_ratios.idxmax()
    best_fast_window, best_slow_window, _ = best_params
    print(f"\nBest Parameters Found (Sharpe):")
    print(f"  Fast Window: {best_fast_window}")
    print(f"  Slow Window: {best_slow_window}")
    print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

    returns = portfolio.returns_stats(agg_func=None)["Total Return [%]"]
    best_params = returns.idxmax()
    best_fast_window, best_slow_window, _ = best_params
    print(f"\nBest Parameters Found (Return):")
    print(f"  Fast Window: {best_fast_window}")
    print(f"  Slow Window: {best_slow_window}")
    print(f"  Resulting ROI: {returns.max():.2f}%")


print("--In Sample Best Params--")
find_best_dual_sma_params(
    in_sample["Close"], min_window_size=2, max_window_size=400, step=2
)
print("\n\n--Out of Sample Best Params--")
find_best_dual_sma_params(
    out_sample["Close"], min_window_size=2, max_window_size=400, step=2
)

--In Sample Best Params--

Best Parameters Found (Sharpe):
  Fast Window: 10
  Slow Window: 76
  Resulting Sharpe Ratio: 1.13

Best Parameters Found (Return):
  Fast Window: 312
  Slow Window: 314
  Resulting ROI: 446.33%


--Out of Sample Best Params--

Best Parameters Found (Sharpe):
  Fast Window: 14
  Slow Window: 400
  Resulting Sharpe Ratio: inf

Best Parameters Found (Return):
  Fast Window: 100
  Slow Window: 232
  Resulting ROI: 82.85%


Now for single SMA...

In [23]:
def find_best_single_sma_param(close, min_window_size=2, max_window_size=200, step=2):
    windows = np.arange(min_window_size, max_window_size + 1, step)
    sma = vbt.MA.run(close, window=windows, short_name="sma")
    price = close.vbt.tile(len(windows))
    price.columns = sma.ma.columns
    entries = price > sma.ma
    exits = price < sma.ma

    portfolio = vbt.Portfolio.from_signals(
        close=close, entries=entries, exits=exits, init_cash=100_000, freq="D"
    )

    sharpe_ratios = portfolio.sharpe_ratio()
    best_param = sharpe_ratios.idxmax()
    best_window, _ = best_param

    print(f"\nBest Parameter Found:")
    print(f"  Window: {best_window}")
    print(f"  Resulting Sharpe Ratio: {sharpe_ratios.max():.2f}")

    returns = portfolio.returns_stats(agg_func=None)["Total Return [%]"]
    best_param = returns.idxmax()
    best_window, _ = best_param
    print(f"\nBest Parameter Found (Return):")
    print(f"  Window: {best_window}")
    print(f"  Resulting ROI: {returns.max():.2f}%")


print("--In Sample Best Params--")
find_best_single_sma_param(
    in_sample["Close"], min_window_size=2, max_window_size=400, step=2
)
print("\n\n--Out of Sample Best Params--")
find_best_single_sma_param(
    out_sample["Close"], min_window_size=2, max_window_size=400, step=2
)

--In Sample Best Params--

Best Parameter Found:
  Window: 298
  Resulting Sharpe Ratio: 1.02

Best Parameter Found (Return):
  Window: 298
  Resulting ROI: 313.43%


--Out of Sample Best Params--

Best Parameter Found:
  Window: 296
  Resulting Sharpe Ratio: 1.68

Best Parameter Found (Return):
  Window: 296
  Resulting ROI: 70.42%


Now that we have the best parameters in sample and out of sample for both of our strategies, let's compare all of them against each other.

In [24]:
# Original
sma_50_entries, sma_50_exits = single_sma_strategy(data["Close"], window=50)
sma_44_entries, sma_44_exits = single_sma_strategy(data["Close"], window=44)
sma_200_entries, sma_200_exits = single_sma_strategy(data["Close"], window=200)
sma_50_100_entries, sma_50_100_exits = dual_sma_strategy(
    data["Close"], fast_window=50, slow_window=100
)
sma_20_200_entries, sma_20_200_exits = dual_sma_strategy(
    data["Close"], fast_window=20, slow_window=200
)

# New
sma_298_entries, sma_298_exits = single_sma_strategy(data["Close"], window=298)
sma_296_entries, sma_296_exits = single_sma_strategy(data["Close"], window=296)
sma_10_76_entries, sma_10_76_exits = dual_sma_strategy(
    data["Close"], fast_window=10, slow_window=76
)
sma_312_314_entries, sma_312_314_exits = dual_sma_strategy(
    data["Close"], fast_window=312, slow_window=314
)
sma_14_400_entries, sma_14_400_exits = dual_sma_strategy(
    data["Close"], fast_window=14, slow_window=400
)
sma_100_232_entries, sma_100_232_exits = dual_sma_strategy(
    data["Close"], fast_window=100, slow_window=232
)

all_entries = pd.concat(
    {
        "Single_SMA_50": sma_50_entries,
        "Single_SMA_44": sma_44_entries,
        "Single_SMA_200": sma_200_entries,
        "Single_SMA_296": sma_296_entries,
        "Single_SMA_298": sma_298_entries,
        "SMA_50_100": sma_50_100_entries,
        "SMA_20_200": sma_20_200_entries,
        "SMA_312_314": sma_312_314_entries,
        "SMA_14_400": sma_14_400_entries,
        "SMA_100_232": sma_100_232_entries,
    },
    axis=1,
)  # axis=1 means to stack horizontally (add more columns)

all_exits = pd.concat(
    {
        "Single_SMA_50": sma_50_exits,
        "Single_SMA_44": sma_44_exits,
        "Single_SMA_200": sma_200_exits,
        "Single_SMA_296": sma_296_exits,
        "Single_SMA_298": sma_298_exits,
        "SMA_50_100": sma_50_100_exits,
        "SMA_20_200": sma_20_200_exits,
        "SMA_312_314": sma_312_314_exits,
        "SMA_14_400": sma_14_400_exits,
        "SMA_100_232": sma_100_232_exits,
    },
    axis=1,
)

in_entries = all_entries.loc[in_sample.index]
in_exits = all_exits.loc[in_sample.index]
out_entries = all_entries.loc[out_sample.index]
out_exits = all_exits.loc[out_sample.index]


print("--IN-SAMPLE PERFORMANCE--")
in_sample_portfolio = vbt.Portfolio.from_signals(
    close=in_sample["Close"],
    entries=in_entries,
    exits=in_exits,
    init_cash=100_000,
    freq="D",
)

display(
    in_sample_portfolio.stats(agg_func=None)
    .sort_values("Total Return [%]", ascending=False)
    .T
)  # Use agg_func=None to stop vectorbt from taking the average of all strategies

print("\n--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---")
out_sample_portfolio = vbt.Portfolio.from_signals(
    close=out_sample["Close"],
    entries=out_entries,
    exits=out_exits,
    init_cash=100_000,
    freq="D",
)

display(
    out_sample_portfolio.stats(agg_func=None)
    .sort_values("Total Return [%]", ascending=False)
    .T
)

--IN-SAMPLE PERFORMANCE--


,SMA_312_314,Single_SMA_298,Single_SMA_296,Single_SMA_200,SMA_14_400,SMA_100_232,SMA_20_200,SMA_50_100,Single_SMA_50,Single_SMA_44
Ticker,SPY,SPY,SPY,SPY,SPY,SPY,SPY,SPY,SPY,SPY
Start,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00,2007-01-03 00:00:00
End,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00,2021-12-31 00:00:00
Period,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00,3777 days 00:00:00
Start Value,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
End Value,546328.906176,413426.627512,404279.712867,357524.670765,336629.580662,334154.754991,329160.961612,299838.435689,197460.794702,195027.345859
Total Return [%],446.328906,313.426628,304.279713,257.524671,236.629581,234.154755,229.160962,199.838436,97.460795,95.027346
Benchmark Return [%],352.008972,352.008972,352.008972,352.008972,352.008972,352.008972,352.008972,352.008972,352.008972,352.008972
Max Gross Exposure [%],100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
Total Fees Paid,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0



--- OUT-OF-SAMPLE PERFORMANCE (Unseen validation data) ---


,SMA_100_232,SMA_14_400,SMA_20_200,Single_SMA_296,Single_SMA_298,SMA_312_314,Single_SMA_200,Single_SMA_44,SMA_50_100,Single_SMA_50
Ticker,SPY,SPY,SPY,SPY,SPY,SPY,SPY,SPY,SPY,SPY
Start,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00,2022-01-03 00:00:00
End,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00,2025-10-27 00:00:00
Period,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00,958 days 00:00:00
Start Value,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0,100000.0
End Value,182853.794868,171517.903489,165879.373344,157976.213364,157976.213364,154801.171067,148624.542271,144960.682235,144405.693097,138658.17509
Total Return [%],82.853795,71.517903,65.879373,57.976213,57.976213,54.801171,48.624542,44.960682,44.405693,38.658175
Benchmark Return [%],50.854894,50.854894,50.854894,50.854894,50.854894,50.854894,50.854894,50.854894,50.854894,50.854894
Max Gross Exposure [%],100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0,100.0
Total Fees Paid,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### Analysis

**1. The Overfitting Problem is Evident**

*  `SMA_312_314`, `Single_SMA_298`, and `Single_SMA_296` were the top 3 in-sample (IS) performers (446%, 313%, 304% returns). Out-of-sample (OOS), they dropped to the middle/bottom of the pack (53%-56% returns).

* Conversely, `SMA_100_232` and `SMA_14_400` were mediocre IS performers (5th and 6th place). OOS, they became the clear winners (1st and 2nd place).

* **Explanation**: The very long and specific windows (like 312/314) were likely optimized to capture the specific bull/bear cycles of 2007-2021 (including the 2008 Financial Crisis and the long bull market that followed). These conditions did not persist into the 2022-2025 period (characterized by high inflation, rapid rate hikes, and a bear market in 2022), so those hyper-specialized strategies failed.

**2. Key Metric Degradation**

* **Profit Factor** Several Strategies (`SMA_312_314`, `SMA_44`) saw their Profit Factor drop, going from very strong (26, 1.69) to weak or terrible (0.87, 1.66). A Profit Factor below 1.0 (`SMA_312_314` at 0.87) means the strategy is losing money on its trades.

* **Win Rate** This is a massive red flag to the validity of our backtesting.

* `SMA_20_200` decreased from 50.0% to 33.3%.

* `SMA_14_400` went from 60% to NaN (0 closed trades).

* This suggests the IS win rates were a product of the specific market regime and are not repeatable.

* It also suggests that the training size was again too short to get any meaningful data from the out-of-sample test. Would a strategy be worth it if it took years of trading to see validity IRL? Additionally, Do we just pull more data? At what point does data get too old to train on?

**3. Parameter Awards**

`SMA_100_232`: **Our best OOS performer.** Highest return, high Sharpe/Sortino. Its IS win rate was suspiciously high, but it still performed well OOS despite that metric normalizing.

`SMA_14_400`: **Our second-best OOS performer.** It had a very long avg winning trade duration IS (717 days), suggesting it's a long-term trend-following system that avoids whipsaws. This worked well in the volatile but ultimately trending market of 2022-2025.

`SMA_20_200`: **Honorable mention.** The classic crossover. It performed solidly in both periods, never being the best but never being the worst. This is a sign of a robust, well-known strategy that isn't over-optimized.

`SMA_312_314`: **THE SURVIVOR.** (goat?) The only strategy that out performed the benchmark (just buying and holding the S&P 500 index 'SPY'). The Win Rate droping to less than 1 is kinda embarassing though. The only reason this startegy ended positive is because of a very large still open position:

In [25]:
out_sample_portfolio.plot(column=('SMA_312_314', 'SPY'))

FigureWidget({
    'data': [{'legendgroup': '0',
              'line': {'color': '#1f77b4'},
              'name': 'Close',
              'showlegend': True,
              'type': 'scatter',
              'uid': '0e5ae62e-fe90-413d-90b5-2efb97e28f76',
              'x': array(['2022-01-03T00:00:00.000000000', '2022-01-04T00:00:00.000000000',
                          '2022-01-05T00:00:00.000000000', ..., '2025-10-23T00:00:00.000000000',
                          '2025-10-24T00:00:00.000000000', '2025-10-27T00:00:00.000000000'],
                         shape=(958,), dtype='datetime64[ns]'),
              'xaxis': 'x',
              'y': {'bdata': ('AAAAoF1TfEAAAAAA8FB8QAAAACC+xX' ... 'CAFP6EQAAAAAAAKoVAAAAAIIVdhUA='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'customdata': {'bdata': ('AAAAAADgZUBaLTlllcNxQAAAAAAAAA' ... 'AAAADgZkDmQ2ybcU1sQAAAAAAAAAAA'),
                             'dtype': 'f8',
                             'shape': '5, 3'},
     

This graph leads to an important observation. Two moving averages with such similar and long windows (312 and 314 days) will almost **never** cross each other.

Think of a 312-day SMA and a 314-day SMA as two enormous, heavy oil tankers sailing side-by-side.

1. They Are Extremely Slow: Both are averaging the price over approximately 1.25 years of trading data. They are massive ships that are very slow to turn and are highly resistant to daily price "waves."

2. They Are Nearly Identical: Because their window lengths are only two days apart, their calculated values at any given time will be almost the same. They are two tankers of almost the exact same size and weight.


### **Next Steps**

Test Your Next Hypothesis on the Champion: Now is the perfect time to apply a Regime Filter. Create a new strategy variation: SMA_20_200_with_Filter.

The Logic: The strategy is only allowed to enter a trade if the SPY price is also above its 200-day moving average.

The Goal: Run this new strategy on your full dataset (2007-present). The specific goal is to see if this filter rule reduces the SMA_20_200's historical Max Drawdown (which was a rather high 34% in-sample) without significantly damaging its total return. If you can cut that drawdown while keeping most of the profit, you have made a definitive improvement.

Or, observe what happens if you download intraday data and use minute SMAs.

Another thing I'm interested in is incorporating exponetial moving average into these tests.